# Classifier-Guided Abstractive Summarization

This notebook compares two **identical BART-large backbones** on the same AMI test meetings:

| Arm | Checkpoint | Training |
|---|---|---|
| Existing model | `philschmid/bart-large-cnn-samsum` | Used unchanged |
| Your model | Same checkpoint | Further fine-tuned on AMI meeting summaries |

Both models receive the same classifier-guided source:

```text
AMI transcript → importance selection → [DECISION] / [ACTION] markers → BART summary
```

This isolates the effect of **your AMI fine-tuning**. SAMSum was already used to create the
starting checkpoint; you do not need to retrain on SAMSum to make this comparison.

**Data**
- AMI: the local `amicorpus/` folder already in this project. It supplies 142
  transcript → human abstract pairs.
- SAMSum reference dataset: https://huggingface.co/datasets/knkarthick/samsum

> Before running in Colab, upload the trained classifier folders to `models/`:
> `importance_classifier.pkl`, `bert_decision/`, and `bert_action_extractor/`.
> The notebook stops with a clear error if any required artifact is missing.

## 1. Setup

In [ ]:
import inspect
import json
import os
import pickle
import random
import re
import subprocess
import sys
import warnings
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers>=4.40", "datasets", "evaluate", "rouge-score",
         "bert-score", "sentencepiece", "accelerate", "scipy"],
        check=True,
    )

import evaluate
import numpy as np
import pandas as pd
import torch
from bert_score import score as bertscore
from datasets import Dataset, DatasetDict
from scipy.sparse import csr_matrix, hstack
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from transformers import (
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "Enable a T4 GPU: Runtime → Change runtime type → T4 GPU."
print(f"Device: {torch.cuda.get_device_name(0)}")

def find_root():
    for candidate in [
        Path("."), Path(".."), Path("/content"), Path("/content/project"),
        Path("/content/drive/MyDrive/Graduation Project"),
    ]:
        if (candidate / "amicorpus" / "abstractive").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find amicorpus/abstractive. Set ROOT to the folder containing amicorpus/."
    )

ROOT = find_root()
AMI_ROOT = ROOT / "amicorpus"
MODELS_DIR = ROOT / "models"
OUTPUT_DIR = MODELS_DIR / "summarization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMPORTANCE_PATH = MODELS_DIR / "importance_classifier.pkl"
DECISION_PATH = MODELS_DIR / "bert_decision"
ACTION_PATH = MODELS_DIR / "bert_action_extractor"

BASELINE_MODEL = "philschmid/bart-large-cnn-samsum"
MAX_SOURCE_TOKENS = 900
MAX_TARGET_TOKENS = 220
CLASSIFIER_BATCH_SIZE = 32

print("Project:", ROOT)
print("Output :", OUTPUT_DIR)

## 2. Load the three classifier artifacts

The importance classifier selects source content. The decision and action models add structural
markers, but do **not** replace importance selection. A selected sentence can receive both markers.

In [ ]:
missing = [p for p in [IMPORTANCE_PATH, DECISION_PATH, ACTION_PATH] if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Upload these trained artifacts to the Colab project models/ folder:\n"
        + "\n".join(f"  - {p}" for p in missing)
    )

with open(IMPORTANCE_PATH, "rb") as f:
    importance_artifact = pickle.load(f)

importance_model = importance_artifact["model"]
importance_vectorizer = importance_artifact["tfidf_vectorizer"]

decision_tokenizer = AutoTokenizer.from_pretrained(DECISION_PATH)
decision_model = AutoModelForSequenceClassification.from_pretrained(DECISION_PATH).to(DEVICE).eval()

action_tokenizer = AutoTokenizer.from_pretrained(ACTION_PATH)
action_model = AutoModelForTokenClassification.from_pretrained(ACTION_PATH).to(DEVICE).eval()
action_id2label = {int(k): v for k, v in action_model.config.id2label.items()}

print("Importance:", importance_artifact["best_model_name"])
print("Decision  :", DECISION_PATH)
print("Action    :", ACTION_PATH)

In [ ]:
# These feature definitions exactly match notebooks/02_importance_classifier.ipynb.
MODAL_VERBS = re.compile(r"\b(will|would|shall|should|must|need to|ought to|going to|have to|need|wants to)\b", re.I)
DECISION_KWS = re.compile(r"\b(decided|agreed|confirmed|resolved|concluded|chosen|approved|rejected|accepted|finalised|finalized|going with|stick with|settle on)\b", re.I)
ACTION_VERBS = re.compile(r"\b(make|create|build|prepare|write|send|review|check|update|implement|design|discuss|look into|find out|contact|set up|organise|organize|schedule|arrange|follow up)\b", re.I)
NEGATION_KWS = re.compile(r"\b(not|no|never|without|don't|doesn't|didn't|won't|wouldn't|can't|cannot)\b", re.I)
FIRST_PERSON = re.compile(r"\b(I|we|our|my|us)\b")
QUESTION_RE = re.compile(r"\?")

def importance_features(texts):
    rows = []
    for text in texts:
        words = text.split()
        n = max(len(words), 1)
        rows.append([
            len(MODAL_VERBS.findall(text)), len(DECISION_KWS.findall(text)),
            len(ACTION_VERBS.findall(text)), len(NEGATION_KWS.findall(text)),
            len(FIRST_PERSON.findall(text)), int(bool(QUESTION_RE.search(text))),
            len(words), len(text), len(MODAL_VERBS.findall(text)) / n,
            sum(w[0].isupper() and len(w) > 1 for w in words) / n,
            int(text.endswith(".")), text.count(",") / n,
        ])
    return np.asarray(rows, dtype=float)

def score_importance(texts):
    """Return P(important) using the saved importance classifier."""
    x_tf = importance_vectorizer.transform(texts)
    # This mirrors the original notebook's inference helper.
    x_hc = StandardScaler().fit_transform(importance_features(texts))
    x = hstack([x_tf, csr_matrix(x_hc)])
    return importance_model.predict_proba(x)[:, 1]

@torch.no_grad()
def score_decisions(texts):
    scores = []
    for start in range(0, len(texts), CLASSIFIER_BATCH_SIZE):
        batch = texts[start:start + CLASSIFIER_BATCH_SIZE]
        enc = decision_tokenizer(batch, truncation=True, padding=True,
                                  max_length=128, return_tensors="pt").to(DEVICE)
        probs = torch.softmax(decision_model(**enc).logits, dim=-1)[:, 1]
        scores.extend(probs.cpu().tolist())
    return np.asarray(scores)

@torch.no_grad()
def detect_actions(texts):
    """True when the BIO extractor predicts at least one ACTION span."""
    flags = []
    for start in range(0, len(texts), CLASSIFIER_BATCH_SIZE):
        batch = texts[start:start + CLASSIFIER_BATCH_SIZE]
        words = [text.split() for text in batch]
        enc = action_tokenizer(words, is_split_into_words=True, truncation=True,
                               padding=True, max_length=128, return_tensors="pt").to(DEVICE)
        pred_ids = action_model(**enc).logits.argmax(dim=-1).cpu().numpy()
        for i, word_list in enumerate(words):
            word_ids = enc.word_ids(batch_index=i)
            seen, labels = set(), []
            for pos, word_id in enumerate(word_ids):
                if word_id is not None and word_id not in seen:
                    seen.add(word_id)
                    labels.append(action_id2label[int(pred_ids[i, pos])])
            flags.append(any(label.endswith("ACTION") for label in labels[:len(word_list)]))
    return np.asarray(flags)

## 3. Parse AMI into speaker-labelled transcript chunks

This checkout contains AMI word XML but not dialogue-act segmentation. The parser reconstructs
speaker chunks by splitting at `.`, `?`, `!`, or after 40 words. This is an approximation; it is
kept identical for train, validation, test, and both model arms.

In [ ]:
TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*|[.?!,;:]")
PUNCTUATION = {".", "?", "!"}

def local_name(element):
    return element.tag.rsplit("}", 1)[-1]

def parse_word_file(path):
    """Return timestamped words from one AMI speaker channel."""
    root = ET.parse(path).getroot()
    words = []
    for elem in root.iter():
        if local_name(elem) != "w" or not (elem.text or "").strip():
            continue
        start = elem.attrib.get("starttime")
        if start is not None:
            words.append((float(start), (elem.text or "").strip()))
    return words

def clean_text(tokens):
    return (" ".join(tokens).replace(" ,", ",").replace(" .", ".")
            .replace(" ?", "?").replace(" !", "!").replace(" ;", ";").replace(" :", ":"))

def parse_meeting(meeting_id, max_words=40):
    """Merge all speaker channels by time, then create chronological pseudo-utterances.

    AMI dialogue-act segmentation is not included in this checkout, so punctuation, speaker
    changes, and a 40-word cap approximate turns. The same parser is used for every split.
    """
    timed_words = []
    for word_file in (AMI_ROOT / "words").glob(f"{meeting_id}.*.words.xml"):
        speaker = word_file.name.split(".")[1]
        timed_words.extend((timestamp, speaker, token)
                           for timestamp, token in parse_word_file(word_file))
    timed_words.sort(key=lambda row: row[0])

    utterances, current_speaker, current_tokens, lexical = [], None, [], 0
    for _, speaker, token in timed_words:
        speaker_changed = current_speaker is not None and speaker != current_speaker
        should_close = speaker_changed and lexical >= 3
        if should_close:
            utterances.append({"speaker": current_speaker, "text": clean_text(current_tokens)})
            current_tokens, lexical = [], 0
        current_speaker = speaker
        current_tokens.append(token)
        lexical += bool(re.search(r"[A-Za-z0-9]", token))
        if token in PUNCTUATION or lexical >= max_words:
            if lexical >= 3:
                utterances.append({"speaker": current_speaker, "text": clean_text(current_tokens)})
            current_speaker, current_tokens, lexical = None, [], 0
    if lexical >= 3:
        utterances.append({"speaker": current_speaker, "text": clean_text(current_tokens)})
    return utterances

def read_abstract(meeting_id):
    path = AMI_ROOT / "abstractive" / f"{meeting_id}.abssumm.xml"
    root = ET.parse(path).getroot()
    section = root.find("abstract")
    return " ".join((sentence.text or "").strip() for sentence in section.findall("sentence")
                    if (sentence.text or "").strip())

meeting_ids = sorted(
    path.name.split(".")[0]
    for path in (AMI_ROOT / "abstractive").glob("*.abssumm.xml")
    if list((AMI_ROOT / "words").glob(f"{path.name.split('.')[0]}.*.words.xml"))
)
print(f"Usable AMI meetings: {len(meeting_ids)}")
print("First meeting:", meeting_ids[0], "chronological chunks:", len(parse_meeting(meeting_ids[0])))
print("Reference:", read_abstract(meeting_ids[0])[:250], "…")

## 4. Classifier-guided source construction

In [ ]:
def pack_lines(lines, tokenizer, max_tokens=MAX_SOURCE_TOKENS):
    """Pack complete lines without exceeding BART's source budget."""
    kept, used = [], 0
    for line in lines:
        n_tokens = len(tokenizer(line, add_special_tokens=False).input_ids)
        if n_tokens > max_tokens:
            continue
        if used + n_tokens > max_tokens:
            break
        kept.append(line)
        used += n_tokens
    return "\n".join(kept), used

def build_raw_source(utterances, tokenizer):
    """Chronological transcript baseline: no classifier filtering or structural markers."""
    lines = [f"{u['speaker']}: {u['text']}" for u in utterances]
    source, used = pack_lines(lines, tokenizer)
    return source, {"chunks": len(source.splitlines()), "source_tokens": used}

def build_guided_source(utterances, tokenizer):
    texts = [u["text"] for u in utterances]
    importance = score_importance(texts)
    decision = score_decisions(texts)
    action = detect_actions(texts)

    ranked = sorted(
        range(len(utterances)),
        key=lambda i: (action[i] or decision[i] >= 0.50, importance[i]),
        reverse=True,
    )
    selected, used = [], 0
    for index in ranked:
        u = utterances[index]
        marker = ""
        if decision[index] >= 0.50:
            marker += "[DECISION] "
        if action[index]:
            marker += "[ACTION] "
        line = f"{marker}{u['speaker']}: {u['text']}"
        n_tokens = len(tokenizer(line, add_special_tokens=False).input_ids)
        if n_tokens <= MAX_SOURCE_TOKENS - used:
            selected.append((index, line, float(importance[index]), bool(action[index]),
                             float(decision[index])))
            used += n_tokens

    selected.sort(key=lambda row: row[0])  # restore chronological coherence after ranking
    return "\n".join(row[1] for row in selected), {
        "chunks": len(selected), "source_tokens": used,
        "decision_chunks": sum(row[4] >= 0.50 for row in selected),
        "action_chunks": sum(row[3] for row in selected),
    }

source_tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL)
added_tokens = source_tokenizer.add_special_tokens(
    {"additional_special_tokens": ["[DECISION]", "[ACTION]"]}
)
example_utterances = parse_meeting(meeting_ids[0])
example_source, example_metadata = build_guided_source(example_utterances, source_tokenizer)
print(f"Added {added_tokens} structural marker tokens")
print(f"Selected {example_metadata['chunks']} chunks / {example_metadata['source_tokens']} source tokens")
print(example_source[:1800])

## 5. Build a meeting-level train / validation / test split

Meetings, never individual chunks, are split. The same pairs are used to train your model and to
evaluate both arms. Generated pairs are saved so subsequent runs do not rerun the three classifiers.

In [ ]:
PAIR_PATH = ROOT / "data" / "processed" / "ami_bart_summary_pairs.json"
SPLIT_PATH = ROOT / "data" / "processed" / "ami_bart_meeting_split.json"
REBUILD_PAIRS = True  # Set False only after confirming the saved pair schema below.

if REBUILD_PAIRS:
    pairs = []
    for n, meeting_id in enumerate(meeting_ids, 1):
        utterances = parse_meeting(meeting_id)
        raw_source, raw_metadata = build_raw_source(utterances, source_tokenizer)
        guided_source, guided_metadata = build_guided_source(utterances, source_tokenizer)
        pairs.append({
            "meeting_id": meeting_id,
            "raw_source": raw_source,
            "guided_source": guided_source,
            "target": read_abstract(meeting_id),
            "raw_metadata": raw_metadata,
            "guided_metadata": guided_metadata,
        })
        if n % 10 == 0:
            print(f"Built {n}/{len(meeting_ids)} meeting pairs")
    PAIR_PATH.parent.mkdir(parents=True, exist_ok=True)
    PAIR_PATH.write_text(json.dumps(pairs, ensure_ascii=False, indent=2), encoding="utf-8")
else:
    pairs = json.loads(PAIR_PATH.read_text(encoding="utf-8"))

# Split entire meetings once and save IDs, so reruns cannot silently change the test set.
if SPLIT_PATH.exists():
    split_ids = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
else:
    train_ids, holdout_ids = train_test_split([p["meeting_id"] for p in pairs], test_size=0.30, random_state=SEED)
    validation_ids, test_ids = train_test_split(holdout_ids, test_size=0.50, random_state=SEED)
    split_ids = {"train": sorted(train_ids), "validation": sorted(validation_ids), "test": sorted(test_ids), "seed": SEED}
    SPLIT_PATH.write_text(json.dumps(split_ids, indent=2), encoding="utf-8")

pairs_by_id = {pair["meeting_id"]: pair for pair in pairs}
train_pairs = [pairs_by_id[mid] for mid in split_ids["train"]]
validation_pairs = [pairs_by_id[mid] for mid in split_ids["validation"]]
test_pairs = [pairs_by_id[mid] for mid in split_ids["test"]]
assert not (set(split_ids["train"]) & set(split_ids["validation"]))
assert not (set(split_ids["train"]) & set(split_ids["test"]))
assert not (set(split_ids["validation"]) & set(split_ids["test"]))
assert len(train_pairs) + len(validation_pairs) + len(test_pairs) == len(pairs)

print(f"Train={len(train_pairs)}  Validation={len(validation_pairs)}  Test={len(test_pairs)}")
print("Saved immutable split:", SPLIT_PATH)
print(pd.DataFrame([p["guided_metadata"] for p in pairs]).describe())

## 6. Tokenization and common evaluation

In [ ]:
def to_seq2seq_dataset(rows, source_key="guided_source"):
    return Dataset.from_list([
        {"meeting_id": row["meeting_id"], "source": row[source_key], "target": row["target"]}
        for row in rows
    ])

raw = DatasetDict({
    "train": to_seq2seq_dataset(train_pairs),
    "validation": to_seq2seq_dataset(validation_pairs),
    "test": to_seq2seq_dataset(test_pairs),
})

def tokenize_pairs(batch):
    model_inputs = source_tokenizer(batch["source"], max_length=MAX_SOURCE_TOKENS, truncation=True)
    labels = source_tokenizer(text_target=batch["target"], max_length=MAX_TARGET_TOKENS, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = raw.map(tokenize_pairs, batched=True, remove_columns=raw["train"].column_names)
rouge = evaluate.load("rouge")

# Fail early if pair construction ever exceeds the intended model limits.
for split_name, dataset in raw.items():
    lengths = [len(source_tokenizer(source, add_special_tokens=True).input_ids) for source in dataset["source"]]
    assert max(lengths) <= MAX_SOURCE_TOKENS + 2, f"{split_name} has an overlong source"

def generate_summaries(model, sources):
    model = model.to(DEVICE).eval()
    predictions = []
    for start in range(0, len(sources), 2):
        batch = sources[start:start + 2]
        inputs = source_tokenizer(batch, return_tensors="pt", padding=True,
                                  truncation=True, max_length=MAX_SOURCE_TOKENS).to(DEVICE)
        with torch.no_grad():
            generated = model.generate(
                **inputs, num_beams=4, length_penalty=2.0,
                min_new_tokens=45, max_new_tokens=MAX_TARGET_TOKENS,
            )
        predictions.extend(source_tokenizer.batch_decode(generated, skip_special_tokens=True))
    return predictions

def evaluate_summaries(predictions, references):
    scores = rouge.compute(predictions=predictions, references=references, use_stemmer=True)
    _, _, f1 = bertscore(predictions, references, lang="en", verbose=True)
    return {
        "ROUGE-1": round(scores["rouge1"] * 100, 2),
        "ROUGE-2": round(scores["rouge2"] * 100, 2),
        "ROUGE-L": round(scores["rougeL"] * 100, 2),
        "BERTScore F1": round(float(f1.mean()) * 100, 2),
    }

STOPWORDS = {"a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "in", "is", "it", "of", "on", "or", "that", "the", "to", "was", "we", "with", "will"}
SIGNAL_RE = re.compile(r"\b(decid|agree|confirm|choos|approv|reject|will|should|must|need to|action)\w*\b", re.I)

def marker_support_audit(prediction, guided_source, minimum_overlap=2):
    """Heuristic audit, not a factuality metric: test whether key generated claims overlap marked input lines."""
    marked_lines = [line for line in guided_source.splitlines() if line.startswith(("[DECISION]", "[ACTION]"))]
    source_tokens = [set(re.findall(r"[a-z]+", line.lower())) - STOPWORDS for line in marked_lines]
    claims = [sentence.strip() for sentence in re.split(r"(?<=[.!?])\s+", prediction) if SIGNAL_RE.search(sentence)]
    if not claims:
        return {"key_claims": 0, "supported_claims": 0, "support_rate": np.nan}
    supported = 0
    for claim in claims:
        claim_tokens = set(re.findall(r"[a-z]+", claim.lower())) - STOPWORDS
        supported += any(len(claim_tokens & tokens) >= minimum_overlap for tokens in source_tokens)
    return {"key_claims": len(claims), "supported_claims": supported, "support_rate": supported / len(claims)}

## 7. Arm A — existing SAMSum model (no training)

In [ ]:
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(BASELINE_MODEL)
baseline_model.resize_token_embeddings(len(source_tokenizer))

test_guided_sources = [row["guided_source"] for row in test_pairs]
test_raw_sources = [row["raw_source"] for row in test_pairs]
test_references = [row["target"] for row in test_pairs]

baseline_guided_predictions = generate_summaries(baseline_model, test_guided_sources)
baseline_raw_predictions = generate_summaries(baseline_model, test_raw_sources)
baseline_guided_metrics = evaluate_summaries(baseline_guided_predictions, test_references)
baseline_raw_metrics = evaluate_summaries(baseline_raw_predictions, test_references)
print("Existing SAMSum BART, classifier-guided input:", baseline_guided_metrics)
print("Existing SAMSum BART, raw chronological input:", baseline_raw_metrics)

## 8. Arm B — your AMI fine-tuning

The model starts from the **same SAMSum checkpoint** as Arm A. It sees only training meetings,
uses validation ROUGE-L for early stopping, and never sees test meetings before final evaluation.

In [ ]:
my_model = AutoModelForSeq2SeqLM.from_pretrained(BASELINE_MODEL)
my_model.resize_token_embeddings(len(source_tokenizer))
my_model.config.use_cache = False  # required with gradient checkpointing

def trainer_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    # Depending on the Transformers version, predictions may be logits
    # or generated IDs padded with negative sentinel values. Normalize both
    # cases before decoding; fast tokenizers reject negative or float IDs.
    predictions = np.asarray(predictions)
    if predictions.ndim == 3:
        predictions = predictions.argmax(axis=-1)
    predictions = np.where(predictions < 0, source_tokenizer.pad_token_id, predictions).astype(np.int64)
    labels = np.where(labels < 0, source_tokenizer.pad_token_id, labels).astype(np.int64)
    decoded_predictions = source_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = source_tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_predictions, references=decoded_labels, use_stemmer=True)
    return {"rouge1": result["rouge1"], "rouge2": result["rouge2"], "rougeL": result["rougeL"]}

training_kwargs = dict(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=5,
    learning_rate=3e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=2,
    gradient_checkpointing=True,
    fp16=True,
    weight_decay=0.01,
    warmup_ratio=0.10,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_TOKENS,
    generation_num_beams=4,
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
    seed=SEED,
)
# Transformers renamed this argument; support both current and older Colab versions.
evaluation_key = "eval_strategy" if "eval_strategy" in inspect.signature(Seq2SeqTrainingArguments).parameters else "evaluation_strategy"
training_kwargs[evaluation_key] = "epoch"
training_args = Seq2SeqTrainingArguments(**training_kwargs)

trainer_kwargs = dict(
    model=my_model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorForSeq2Seq(source_tokenizer, model=my_model, label_pad_token_id=-100),
    compute_metrics=trainer_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
# Newer Transformers renamed the tokenizer= argument to processing_class=.
tokenizer_key = "processing_class" if "processing_class" in inspect.signature(Seq2SeqTrainer).parameters else "tokenizer"
trainer_kwargs[tokenizer_key] = source_tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)
trainer.train()
trainer.save_model(OUTPUT_DIR / "classifier_guided_bart_ami")
source_tokenizer.save_pretrained(OUTPUT_DIR / "classifier_guided_bart_ami")

## 9. Final validation: existing model vs your fine-tuned model

In [ ]:
fine_tuned_guided_predictions = generate_summaries(trainer.model, test_guided_sources)
fine_tuned_raw_predictions = generate_summaries(trainer.model, test_raw_sources)
fine_tuned_guided_metrics = evaluate_summaries(fine_tuned_guided_predictions, test_references)
fine_tuned_raw_metrics = evaluate_summaries(fine_tuned_raw_predictions, test_references)

comparison = pd.DataFrame(
    [baseline_guided_metrics, fine_tuned_guided_metrics, fine_tuned_raw_metrics, baseline_raw_metrics],
    index=[
        "SAMSum BART — classifier-guided input",
        "AMI fine-tuned BART — classifier-guided input",
        "AMI fine-tuned BART — raw chronological input",
        "SAMSum BART — raw chronological input",
    ],
)
# Force readable text because Colab dark mode can render highlighted values invisible.
styled_comparison = (
    comparison.style
    .set_properties(**{"color": "#ffffff"})
    .highlight_max(axis=0, props="background-color: #c6efce; color: #000000; font-weight: bold;")
    .format("{:.2f}")
)
display(styled_comparison)

# The audit is deliberately reported separately: token overlap is a screening aid, not proof of factuality.
audit = pd.DataFrame([
    marker_support_audit(prediction, pair["guided_source"])
    for prediction, pair in zip(fine_tuned_guided_predictions, test_pairs)
])
audit_summary = audit[["key_claims", "supported_claims", "support_rate"]].mean(numeric_only=True).to_dict()
print("Fine-tuned guided marker-support audit (heuristic):", audit_summary)

results_path = OUTPUT_DIR / "test_metrics.csv"
comparison.to_csv(results_path)
examples = pd.DataFrame({
    "meeting_id": [pair["meeting_id"] for pair in test_pairs],
    "reference": test_references,
    "guided_source": test_guided_sources,
    "raw_source": test_raw_sources,
    "baseline_guided": baseline_guided_predictions,
    "fine_tuned_guided": fine_tuned_guided_predictions,
    "fine_tuned_raw": fine_tuned_raw_predictions,
    "baseline_raw": baseline_raw_predictions,
    "key_claims": audit["key_claims"],
    "supported_claims": audit["supported_claims"],
    "marker_support_rate": audit["support_rate"],
})
examples_path = OUTPUT_DIR / "test_generations.csv"
examples.to_csv(examples_path, index=False)
(OUTPUT_DIR / "marker_support_audit.json").write_text(json.dumps(audit_summary, indent=2), encoding="utf-8")
print("Saved:", results_path)
print("Saved:", examples_path)

In [ ]:
# Qualitative review: inspect the same meeting with both models and both input variants.
for i in range(min(3, len(test_pairs))):
    print("=" * 100)
    print("MEETING:", test_pairs[i]["meeting_id"])
    print("\nREFERENCE:\n", test_references[i])
    print("\nSAMSUM BART — GUIDED:\n", baseline_guided_predictions[i])
    print("\nAMI FINE-TUNED BART — GUIDED:\n", fine_tuned_guided_predictions[i])
    print("\nAMI FINE-TUNED BART — RAW:\n", fine_tuned_raw_predictions[i])
    print("\nHEURISTIC MARKER SUPPORT:\n", audit.iloc[i].to_dict())

## 10. Interpretation and downloads

Use the four-row table to answer two separate questions:

1. **Does AMI fine-tuning help?** Compare the two *classifier-guided* rows. Claim improvement only when the AMI-fine-tuned model improves held-out ROUGE-L and BERTScore.
2. **Do the three classifiers help the summary input?** Compare the two AMI-fine-tuned rows. This is an ablation: raw chronological input versus classifier-filtered, `[DECISION]` / `[ACTION]`-tagged input.

`marker_support_audit.json` is a **heuristic screening result**, not a factuality metric. Review the rows in `test_generations.csv`, especially any key claims without marker support, before making claims about faithfulness.

Download from `models/summarization/` after the final run:
- `classifier_guided_bart_ami/` — the best checkpoint and tokenizer
- `test_metrics.csv` — all four aggregate comparisons
- `test_generations.csv` — test sources, references, predictions, and audit values

Limits to state in the thesis:
- 142 meetings is a small fine-tuning dataset; report the saved split IDs and random seed.
- Transcript chunks are reconstructed from word XML because dialogue-act segmentation is absent.
- The importance model was trained on AMI summary-style text, not raw spoken utterances. A future version should train it using AMI extractive annotations.
- AMI does not provide direct transcript-to-summary alignment, so source selection is weakly supervised.
- This evaluates the full classifier-guided summarization pipeline, not BART in isolation.